# 암종 분류 v6 (v3 + 쌍둥이 규칙 + 표기 통일): 이진 행렬 + 파생변수 + 인프레임 핫스팟 + 동반변이 콤보 + XGB/LR 앙상블 (5-fold CV)

**v2 대비 변경점**
- 인프레임 변이(`EGFR E746_A750del` 등)를 핫스팟 토큰으로 추가 (`gene|inframe`, `gene|위치_inframe`)
- 가이드 기반 동반변이 콤보 피처 (`cb_*`), TMB 기준 5 / 30
- **EM 사전확률 보정 제거** (macroF1을 0.46 -> 0.21로 망가뜨림). 대신 COAD 하나만 약하게 키우는 보정을 시뮬레이션으로 검증 후 자동 적용(이득 없으면 미적용)
- 실행 순서 버그 수정: 예전 prior 셀이 CV보다 앞에 있어 `oof_ens`가 정의되기 전에 쓰였음 -> 앙상블 이후로 이동
- CV를 함수(`run_cv`)로 묶고, 콤보 ON/OFF 비교(ablation) 셀 추가

**패키지**: `pip install pandas numpy scikit-learn xgboost`


**v5 변경점**: 쌍둥이(형제 라벨) 규칙 추가 (5-2절). KIRC↔KIPAN, LGG↔GBMLGG 는 같은 환자가 두 코호트에 중복 등록된 경우가 있어, train에 짝 라벨이 있는 프로필이 test에 나오면 짝 암종으로 교정한다. (최소빈도는 v3 그대로 5)

**v6 변경점**: 변이 표기 통일(`canon`) + 한 칸 안 중복 표기 제거 (`NORMALIZE_TOKENS`). test 일부 행이 `R1450X`, 한 칸에 같은 변이를 여러 표기로 적는 방식이라 변이 수/유형 카운트가 부풀려지는 문제를 막는다. train에는 거의 영향이 없어 CV 점수는 비슷하게 나오는 게 정상이다.

**v7 변경점** (논문 *Advances in ML for tumour classification in CUP* 의 기법 반영)
- **ExtraTrees 추가** (`USE_ET`): 논문 리뷰 모델 대부분이 RF 계열 -> XGB(부스팅)·LR(선형)과 다른 성격의 모델을 3번째 앙상블 멤버로 추가
- **확률 보정** (`CALIB_METHODS`): CUPLR의 isotonic 보정 아이디어. none / temp(온도 스케일링) / isotonic 을 *교차 적합(cross-fit) OOF*에서 비교해 이득이 `BOOST_MIN_GAIN` 이상일 때만 적용
- 앙상블 가중치를 3-way(XGB/LR/ET) simplex 격자(0.1 간격)로 탐색
- `run_cv`가 dict(`oof`, `te`)를 반환 (`oof_xgb`, `oof_lr`, `oof_et` 별칭은 유지)


In [1]:
import os, re, warnings
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, MaxAbsScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
warnings.filterwarnings("ignore")

CFG = dict(
    DATA_DIR=os.environ.get("DATA_DIR", "../../data"),
    SEED=42,
    N_FOLDS=int(os.environ.get("N_FOLDS", 5)),
    MIN_GENE_FREQ=5,        # train에서 변이 샘플이 이 값 미만인 유전자는 제외
    MIN_HOTSPOT_FREQ=5,     # gene|position 토큰 최소 빈도
    MIN_AA_FREQ=20,         # 아미노산 치환쌍 최소 빈도
    USE_AA_SUBS=True,       # 아미노산 치환 패턴 피처 on/off (CV로 효과 확인)
    USE_PATHWAYS=True,      # 경로 집계 피처 on/off
    USE_HOTSPOTS=True,      # gene|position, gene|trunc 토큰 on/off
    BALANCE=True,           # 클래스 불균형 보정 (sample weight / class_weight)
    XGB_N_EST=int(os.environ.get("XGB_N_EST", 2000)),
    LR_C=0.1,
    USE_COMBOS=True,        # 동반 변이 콤보 피처 on/off
    TMB_LOW=5,              # 극저변이 기준 (THCA/LAML 등)
    TMB_HIGH=30,            # 고변이 기준 (SKCM/UCEC/STES 등)
    COAD_TEST_PRIOR=0.22,   # 가이드가 말하는 test COAD 비율 (검증 시뮬레이션용)
    BOOST_MIN_GAIN=0.003,   # COAD 가중으로 시뮬 macroF1이 이만큼 안 오르면 적용 안 함
    NORMALIZE_TOKENS=True,  # 변이 표기 통일(p./3글자 코드/X->*) + 한 칸 안 중복 표기 제거 (test 표기 차이 대응)
    USE_TWIN=True,          # 쌍둥이(형제 라벨) 규칙: train에 짝 암종 프로필이 있으면 test를 짝 암종으로 교정
    USE_ET=True,            # ExtraTrees(배깅 계열) 앙상블 멤버 추가
    ET_N_EST=500,
    CALIB_METHODS=("none", "temp", "isotonic"),  # 후보 보정 방식 (cross-fit OOF로 비교, 이득 없으면 none)
    RUN_ABLATION=False,     # True면 콤보 ON/OFF CV 비교 (시간 2배)
)


## 1. 데이터 로드

In [2]:
D = CFG["DATA_DIR"]
train = pd.read_csv(f"{D}/train.csv").fillna("WT")
test = pd.read_csv(f"{D}/test.csv").fillna("WT")
sub = pd.read_csv(f"{D}/sample_submission.csv")

assert (test["ID"].values == sub["ID"].values).all(), "test와 sample_submission의 ID 순서가 다릅니다"
gene_cols = [c for c in train.columns if c not in ("ID", "SUBCLASS")]
assert gene_cols == [c for c in test.columns if c != "ID"], "train/test 유전자 컬럼이 다릅니다"

le = LabelEncoder()
y = le.fit_transform(train["SUBCLASS"])
n_class = len(le.classes_)
print("train", train.shape, "test", test.shape, "classes", n_class)
print(train["SUBCLASS"].value_counts())

train (6201, 4386) test (2546, 4385) classes 26
SUBCLASS
BRCA      786
KIPAN     515
GBMLGG    461
STES      379
KIRC      334
THCA      324
SKCM      276
PRAD      266
OV        253
LGG       229
HNSC      223
COAD      223
SARC      198
UCEC      198
LUAD      184
LUSC      178
LIHC      158
LAML      158
CESC      155
PCPG      147
TGCT      124
PAAD      120
BLCA      104
THYM       98
ACC        72
DLBC       38
Name: count, dtype: int64


In [3]:
train.head(5)

,ID,SUBCLASS,A2M,AAAS,AADAT,AARS1,ABAT,ABCA1,ABCA2,ABCA3,...,ZNF292,ZNF365,ZNF639,ZNF707,ZNFX1,ZNRF4,ZPBP,ZW10,ZWINT,ZYX
0,TRAIN_0000,KIPAN,WT,WT,WT,WT,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT
1,TRAIN_0001,SARC,WT,WT,WT,WT,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT
2,TRAIN_0002,SKCM,R895R,WT,WT,WT,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT
3,TRAIN_0003,KIRC,WT,WT,WT,WT,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT
4,TRAIN_0004,GBMLGG,WT,WT,WT,WT,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT


## EDA

In [4]:
import pandas as pd
import numpy as np
import re

# 1. 데이터 로드
train = pd.read_csv(f"{D}/train.csv").fillna("WT")
test = pd.read_csv(f"{D}/test.csv").fillna("WT")

print("="*50)
print("1. 기본 데이터 크기 및 암종(SUBCLASS) 개수 확인")
print("="*50)
print(f"Train 크기: {train.shape}")
print(f"Test 크기: {test.shape}")

# 암종(SUBCLASS) 개수 및 분포 확인
subclass_counts = train["SUBCLASS"].value_counts()
print(f"총 암종(Subclass) 개수: {len(subclass_counts)}개")
print("\n[상위 5개 암종 분포]")
print(subclass_counts.head())

# 유전자 컬럼 분리
gene_cols = [c for c in train.columns if c not in ("ID", "SUBCLASS")]
print(f"\n유전자(Feature) 컬럼 총 개수: {len(gene_cols)}개")


print("\n" + "="*50)
print("2. 샘플당 변이 유전자 수 분포 (TMB)")
print("="*50)
# WT가 아닌 유전자의 개수를 행별로 카운트
mut_matrix = (train[gene_cols] != "WT")
mut_counts_per_sample = mut_matrix.sum(axis=1)
print(mut_counts_per_sample.describe().round(2))


print("\n" + "="*50)
print("3. 한 셀에 변이가 여러 개 들어있는지 확인 (구분자 체크)")
print("="*50)
# WT가 아닌 셀들만 추출
all_vals = train[gene_cols].values.ravel()
non_wt_vals = all_vals[all_vals != "WT"]

non_wt_series = pd.Series(non_wt_vals)
# 쉼표(,), 콜론(:), 세미콜론(;), 파이프(|), 공백 등이 포함된 비율 확인
sep_pattern = r"[,:;|\s]"
has_sep = non_wt_series.str.contains(sep_pattern, regex=True).mean()
print(f"Non-WT 값 중 구분자(, : ; | 공백)가 포함된 비율: {has_sep:.4%}")


print("\n" + "="*50)
print("4. 변이 표기 패턴 종류 확인 (정규식 기반 상위 패턴)")
print("="*50)
# 숫자를 'N'으로 치환하여 대표적인 변이 표기 패턴 분석
pattern_samples = non_wt_series.str.replace(r"\d+", "N", regex=True)
print(pattern_samples.value_counts().head(15))


print("\n" + "="*50)
print("5. 유전자별 변이 빈도 (희소성 확인)")
print("="*50)
gene_mut_freq = mut_matrix.sum(axis=0)
print(f"변이가 단 1개라도 발생한 유전자 수: {(gene_mut_freq > 0).sum()} / {len(gene_cols)}")
print(f"대다수 유전자의 변이 빈도 요약 (상위/하위):")
print(gene_mut_freq.describe().round(2))

1. 기본 데이터 크기 및 암종(SUBCLASS) 개수 확인
Train 크기: (6201, 4386)
Test 크기: (2546, 4385)
총 암종(Subclass) 개수: 26개

[상위 5개 암종 분포]
SUBCLASS
BRCA      786
KIPAN     515
GBMLGG    461
STES      379
KIRC      334
Name: count, dtype: int64

유전자(Feature) 컬럼 총 개수: 4384개

2. 샘플당 변이 유전자 수 분포 (TMB)
count    6201.00
mean       35.30
std       100.78
min         0.00
25%         7.00
50%        14.00
75%        27.00
max      2393.00
dtype: float64

3. 한 셀에 변이가 여러 개 들어있는지 확인 (구분자 체크)
Non-WT 값 중 구분자(, : ; | 공백)가 포함된 비율: 10.0625%

4. 변이 표기 패턴 종류 확인 (정규식 기반 상위 패턴)
LNL    7409
ENK    6790
SNS    4517
RNH    4441
RNC    4141
RNQ    4127
PNP    4037
ANA    3935
TNT    3841
ANT    3712
DNN    3685
INI    3680
ANV    3652
GNG    3581
PNS    3536
Name: count, dtype: int64

5. 유전자별 변이 빈도 (희소성 확인)
변이가 단 1개라도 발생한 유전자 수: 4230 / 4384
대다수 유전자의 변이 빈도 요약 (상위/하위):
count    4384.00
mean       49.93
std        57.64
min         0.00
25%        20.00
50%        36.00
75%        61.00
max      1770.00
dtype: float64


## 2. 변이 표기 형식 확인 (파싱 규칙이 맞는지 눈으로 검증)

In [5]:
sample = pd.Series(train[gene_cols].iloc[:1000].to_numpy(dtype=object).ravel())
nonwt = sample[sample != "WT"]
print(f"non-WT 비율: {len(nonwt)/len(sample):.4%}")
print("한 셀에 구분자(, : ; | 공백)가 있는 비율:", nonwt.str.contains(r"[,:;|\s]").mean().round(4))
print(nonwt.str.replace(r"\d+", "N", regex=True).value_counts().head(25))

non-WT 비율: 0.8337%
한 셀에 구분자(, : ; | 공백)가 있는 비율: 0.0873
LNL    1236
ENK    1125
SNS     777
RNH     752
ANA     738
RNC     724
PNP     711
RNQ     696
TNT     665
ANT     642
GNG     636
ANV     629
FNF     614
PNL     610
DNN     609
PNS     593
INI     561
VNV     555
RNR     492
RNW     483
SNF     437
RN*     410
EN*     388
GNR     341
KNN     328
Name: count, dtype: int64


## 3. 피처 생성

- `g_*` : 유전자별 변이 여부 (이진)
- TMB 계열 : 변이 유전자 수, log, 이벤트 수
- 변이 유형 : missense / silent / nonsense / frameshift / splice / inframe 개수와 비율
- `hs_*` : 핫스팟(`유전자|원래아미노산+위치`) 및 truncating(`유전자|trunc`) 이진
- `aa_*` : 아미노산 치환쌍 비율
- `pw_*` : 경로별 변이 유전자 수 (아래 `PATHWAYS` 사전은 초안이니 자유롭게 수정)

In [6]:
def combo_features(ex):
    mut = ex["mut"]; n = mut.shape[0]
    n_genes = mut.sum(1)
    low, high = n_genes <= CFG["TMB_LOW"], n_genes >= CFG["TMB_HIGH"]

    def G(name):   # 유전자 변이 여부 (없는 유전자는 0 벡터 — 엉뚱한 컬럼 안 가져옴)
        j = gene_pos.get(name)
        return mut[:, j].copy() if j is not None else np.zeros(n, bool)

    tok_rows = {}
    for r, nm in zip(ex["tok_rows"], ex["tok_names"]):
        tok_rows.setdefault(nm, []).append(r)
    def T(name):   # 핫스팟/절단형 토큰 보유 여부 (MIN_HOTSPOT_FREQ와 무관하게 직접 계산)
        v = np.zeros(n, bool)
        if name in tok_rows: v[tok_rows[name]] = True
        return v

    f = {}
    # COAD
    f["COAD_APCtr_TP53"]  = T("APC|trunc") & G("TP53")
    f["COAD_APCtr_KRAS"]  = T("APC|trunc") & (G("KRAS") | G("FBXW7"))
    f["COAD_score"]       = T("APC|trunc").astype(int) + G("TP53") + G("KRAS") + G("FBXW7") + G("PIK3CA")
    # KIRC / KIPAN
    f["KIRC_VHL_PBRM1"]   = G("VHL") & G("PBRM1")
    f["KIRC_3p_score"]    = G("VHL").astype(int) + G("PBRM1") + G("SETD2") + G("BAP1")
    # UCEC
    f["UCEC_PTEN_CTNNB1"] = G("PTEN") & G("CTNNB1")
    f["UCEC_PTEN_PIK3CA"] = G("PTEN") & G("PIK3CA")
    f["UCEC_highTMB_PTEN"]= high & G("PTEN")
    # PAAD
    f["PAAD_score"]       = T("KRAS|G12").astype(int) + G("TP53") + T("CDKN2A|trunc") + G("SMAD4")
    # LGG vs GBMLGG
    f["LGG_IDH1_TP53_ATRX"] = T("IDH1|R132") & G("TP53") & G("ATRX")
    f["LGG_IDH1_clean"]   = T("IDH1|R132") & ~(G("PTEN") | G("EGFR"))
    f["GBM_PTEN_EGFR"]    = G("PTEN") & G("EGFR")
    f["GBM_noIDH1_signal"]= (G("PTEN") | G("EGFR")) & ~G("IDH1")
    # BRAF V600 + TMB (THCA vs SKCM vs COAD)
    f["THCA_BRAF_lowTMB"] = T("BRAF|V600") & low
    f["SKCM_BRAF_highTMB"]= T("BRAF|V600") & high
    f["SKCM_NRAS_highTMB"]= T("NRAS|Q61") & high
    f["COAD_BRAF_APC"]    = T("BRAF|V600") & T("APC|trunc")
    # 폐암 / 위식도 / 간 / 난소 / 백혈병
    f["LUAD_EGFR_noTP53"] = G("EGFR") & ~G("TP53")
    f["LUSC_TP53_noEGFR_highTMB"] = G("TP53") & ~G("EGFR") & high
    f["STES_highTMB_TP53_noAPC"]  = high & G("TP53") & ~T("APC|trunc")
    f["LIHC_CTNNB1_noAPC"]= G("CTNNB1") & ~G("APC")
    f["OV_TP53_lowTMB"]   = G("TP53") & (n_genes <= 20)
    f["LAML_NPM1_lowTMB"] = G("NPM1") & low

    names = list(f.keys())
    return np.c_[[np.asarray(f[k], np.float32) for k in names]].T, names

In [7]:
AA = "ACDEFGHIKLMNPQRSTVWY"
AA_IDX = {a: i for i, a in enumerate(AA)}
TYPES = ["missense", "silent", "nonsense", "frameshift", "splice", "inframe", "other"]
TYPE_IDX = {t: i for i, t in enumerate(TYPES)}
SEP = re.compile(r"[,:;|\s]+")
AA3 = {"Ala":"A","Arg":"R","Asn":"N","Asp":"D","Cys":"C","Gln":"Q","Glu":"E","Gly":"G","His":"H","Ile":"I",
       "Leu":"L","Lys":"K","Met":"M","Phe":"F","Pro":"P","Ser":"S","Thr":"T","Trp":"W","Tyr":"Y","Val":"V","Ter":"*"}

def canon(tok):
    """변이 표기 1개를 한 가지 모양으로 통일한다 (그 토큰만 보고 바꿈 = 행 독립, train/test 동일 적용).
    p.Arg1450Ter / R1450X / R1450* -> R1450*   (옛 HGVS의 X도 종결 코돈으로 읽는 기존 동작 유지)"""
    t = tok.strip()
    if t.startswith("p."):
        t = t[2:]
    t = re.sub(r"[A-Z][a-z]{2}", lambda m: AA3.get(m.group(), m.group()), t)   # 3글자 아미노산 -> 1글자
    t = re.sub(r"^([A-Z])(\d+)X$", r"\1\2*", t)                                  # R1450X -> R1450*
    return t

def cell_tokens(cell):
    """셀 하나 -> 변이 토큰 목록. NORMALIZE_TOKENS면 표기를 통일하고 같은 변이의 중복 표기를 하나로 합친다."""
    toks = [t for t in SEP.split(str(cell)) if t and t != "WT"]
    if CFG["NORMALIZE_TOKENS"]:
        toks = list(dict.fromkeys(canon(t) for t in toks))
    return toks
RE_MIS = re.compile(r"^(?:p\.)?([A-Z])(\d+)([A-Z])$")
RE_NON = re.compile(r"^(?:p\.)?[A-Z]\d+(\*|X|Ter)$")

def parse_token(tok):
    """변이 표기 1개 -> (유형, 원래aa, 위치, 바뀐aa). 표기가 다르면 여기 규칙을 수정하세요."""
    low = tok.lower()
    m = re.search(r"\d+", tok)
    pos = m.group() if m else ""
    if "fs" in low or "frameshift" in low:
        return "frameshift", "", pos, ""
    if "splice" in low:
        return "splice", "", pos, ""
    if "del" in low or "ins" in low or "dup" in low:
        return "inframe", "", pos, ""
    if RE_NON.match(tok):
        return "nonsense", "", pos, ""
    m = RE_MIS.match(tok)
    if m:
        fa, p, ta = m.groups()
        return ("silent" if fa == ta else "missense"), fa, p, ta
    return "other", "", pos, ""

def extract(df):
    arr = df[gene_cols].to_numpy(dtype=object)
    mut = arr != "WT"
    n = len(df)
    type_cnt = np.zeros((n, len(TYPES)), np.float32)
    aa_cnt = np.zeros((n, 400), np.float32)
    tok_rows, tok_names = [], []
    rr, cc = np.nonzero(mut)
    for r, c in zip(rr, cc):
        gene = gene_cols[c]
        for tok in cell_tokens(arr[r, c]):
            typ, fa, pos, ta = parse_token(tok)
            type_cnt[r, TYPE_IDX[typ]] += 1
            if typ == "missense":
                tok_rows.append(r); tok_names.append(f"{gene}|{fa}{pos}")
                if fa in AA_IDX and ta in AA_IDX:
                    aa_cnt[r, AA_IDX[fa] * 20 + AA_IDX[ta]] += 1
            elif typ in ("nonsense", "frameshift", "splice"):
                tok_rows.append(r); tok_names.append(f"{gene}|trunc")
            elif typ == "inframe":                       # ← 추가
                tok_rows.append(r); tok_names.append(f"{gene}|inframe")        # 유전자 단위
                tok_rows.append(r); tok_names.append(f"{gene}|{pos}_inframe")  # 위치 단위 (EGFR 746_inframe 등)
    return dict(mut=mut, type_cnt=type_cnt, aa_cnt=aa_cnt,
                tok_rows=np.array(tok_rows, dtype=np.int64), tok_names=tok_names)

ex_tr, ex_te = extract(train), extract(test)
print("파싱 완료. 유형별 총 개수(train):", dict(zip(TYPES, ex_tr["type_cnt"].sum(0).astype(int))))
# ---- 표기 통일 동작 확인 ----
assert canon("R1450X") == "R1450*" == canon("p.Arg1450Ter") == canon("p.R1450*")
assert canon("E746_A750del") == "E746_A750del" and canon("S221Kfs") == "S221Kfs"
assert parse_token(canon("R1450X"))[0] == "nonsense"
_save = CFG["NORMALIZE_TOKENS"]; CFG["NORMALIZE_TOKENS"] = True
assert cell_tokens("R1450X R1450* p.Arg1450Ter") == ["R1450*"]          # 같은 변이의 여러 표기 -> 1개
assert cell_tokens("R175H, V600E") == ["R175H", "V600E"] and cell_tokens("WT") == []
CFG["NORMALIZE_TOKENS"] = _save


파싱 완료. 유형별 총 개수(train): {'missense': np.int64(161051), 'silent': np.int64(64844), 'nonsense': np.int64(13080), 'frameshift': np.int64(9767), 'splice': np.int64(0), 'inframe': np.int64(3), 'other': np.int64(319)}


### 표기 점검 (참고용 출력만, 이 숫자를 보고 설정을 바꾸지 않는다)
칸당 변이 수가 표기 통일 전/후로 어떻게 달라지는지 본다. test 행 단위가 아니라 전체 평균만 비교하는 **진단용**이다.


In [8]:
def tokens_per_cell(df, normalize):
    vals = df[gene_cols].to_numpy(dtype=object).ravel()
    vals = vals[vals != "WT"]
    n = [len([t for t in SEP.split(str(v)) if t and t != "WT"]) if not normalize else
         len(dict.fromkeys(canon(t) for t in SEP.split(str(v)) if t and t != "WT")) for v in vals]
    return float(np.mean(n)) if n else 0.0

for nm, df in (("train", train), ("test", test)):
    print(f"{nm}: 칸당 변이 수 {tokens_per_cell(df, False):.2f} -> (표기 통일·중복 제거 후) {tokens_per_cell(df, True):.2f}")


train: 칸당 변이 수 1.17 -> (표기 통일·중복 제거 후) 1.14
test: 칸당 변이 수 1.70 -> (표기 통일·중복 제거 후) 1.70


In [9]:
# ---- 경로/유전자군 사전 (초안: 필요에 맞게 수정/추가) ----
PATHWAYS = {
    "p53": ["TP53","MDM2","MDM4","CDKN2A","ATM","CHEK1","CHEK2"],
    "RTK_RAS": ["EGFR","ERBB2","ERBB3","ERBB4","FGFR1","FGFR2","FGFR3","FGFR4","MET","ALK","RET","KIT",
                "PDGFRA","KRAS","NRAS","HRAS","BRAF","RAF1","NF1","MAP2K1","MAP2K2","SOS1","PTPN11"],
    "PI3K": ["PIK3CA","PIK3CB","PIK3R1","PTEN","AKT1","AKT2","AKT3","MTOR","TSC1","TSC2","STK11","RICTOR","INPP4B"],
    "Wnt": ["APC","CTNNB1","AXIN1","AXIN2","RNF43","ZNRF3","TCF7L2","GSK3B","AMER1"],
    "CellCycle": ["RB1","CDK4","CDK6","CCND1","CCND2","CCND3","CCNE1","CDKN1A","CDKN1B","CDKN2B","E2F1","FBXW7"],
    "Notch": ["NOTCH1","NOTCH2","NOTCH3","NOTCH4","FBXW7","SPEN"],
    "Hippo": ["NF2","LATS1","LATS2","SAV1","YAP1","WWTR1","TEAD1"],
    "TGFb": ["TGFBR1","TGFBR2","SMAD2","SMAD3","SMAD4","ACVR1B","ACVR2A"],
    "NRF2": ["NFE2L2","KEAP1","CUL3"],
    "SWI_SNF": ["ARID1A","ARID1B","ARID2","SMARCA4","SMARCB1","PBRM1","SMARCA2"],
    "Chromatin": ["KMT2A","KMT2B","KMT2C","KMT2D","EZH2","SETD2","KDM6A","KDM5C","CREBBP","EP300",
                  "BAP1","ASXL1","DNMT3A","IDH1","IDH2","ATRX"],
    "DDR_MMR": ["BRCA1","BRCA2","PALB2","ATR","POLE","POLD1","MLH1","MSH2","MSH6","PMS2",
                "RAD51","RAD50","MRE11","FANCA","FANCD2","ERCC2"],
    "Splicing": ["SF3B1","SRSF2","U2AF1","RBM10"],
    "LongGenes": ["TTN","MUC16","MUC4","SYNE1","OBSCN","RYR2","RYR1","CSMD1","CSMD3","LRP1B","USH2A","FLG","PCLO"],
}
gene_pos = {g: i for i, g in enumerate(gene_cols)}
PW_IDX = {k: [gene_pos[g] for g in v if g in gene_pos] for k, v in PATHWAYS.items()}
print({k: f"{len(v)}/{len(PATHWAYS[k])}" for k, v in PW_IDX.items()})   # 데이터에 존재하는 유전자 수/사전 크기

{'p53': '5/7', 'RTK_RAS': '14/23', 'PI3K': '11/13', 'Wnt': '6/9', 'CellCycle': '12/12', 'Notch': '5/6', 'Hippo': '4/7', 'TGFb': '6/7', 'NRF2': '2/3', 'SWI_SNF': '0/7', 'Chromatin': '6/16', 'DDR_MMR': '11/16', 'Splicing': '2/4', 'LongGenes': '4/13'}


In [10]:
# ---- train 기준으로만 사용할 유전자/토큰/아미노산쌍 결정 (test는 보지 않음: Data Leakage 방지) ----
# test는 아래 compose()에서 이 기준에 맞춰 변환만 한다. test에만 있는 토큰은 tok_idx.get()이 None이라 무시됨.
freq = ex_tr["mut"].sum(0)
keep_idx = np.where(freq >= CFG["MIN_GENE_FREQ"])[0]

def uniq_pairs(ex):
    return set(zip(ex["tok_rows"].tolist(), ex["tok_names"]))

tok_cnt = Counter(nm for _, nm in uniq_pairs(ex_tr))
keep_tokens = sorted(t for t, c in tok_cnt.items() if c >= CFG["MIN_HOTSPOT_FREQ"])
tok_idx = {t: i for i, t in enumerate(keep_tokens)}

aa_tot = ex_tr["aa_cnt"].sum(0)
aa_keep = np.where(aa_tot >= CFG["MIN_AA_FREQ"])[0]
print(f"유전자 {len(keep_idx)}/{len(gene_cols)}, 핫스팟 토큰 {len(keep_tokens)}, 아미노산쌍 {len(aa_keep)}")

def compose(ex):
    mut = ex["mut"]; n = mut.shape[0]
    blocks, names = [], []
    def add(a, nm):
        blocks.append(np.asarray(a, np.float32).reshape(n, -1)); names.extend(nm)

    add(mut[:, keep_idx], [f"g_{gene_cols[i]}" for i in keep_idx])

    n_genes = mut.sum(1)
    tc = ex["type_cnt"]; n_ev = tc.sum(1)
    add(np.c_[n_genes, np.log1p(n_genes), n_ev], ["n_mut_genes", "log_n_mut_genes", "n_events"])
    add(tc, [f"cnt_{t}" for t in TYPES])
    trunc = tc[:, [TYPE_IDX[t] for t in ("nonsense", "frameshift", "splice")]].sum(1)
    add(np.c_[tc[:, TYPE_IDX["silent"]] / (n_ev + 1),
              tc[:, TYPE_IDX["missense"]] / (n_ev + 1),
              trunc / (n_ev + 1)], ["ratio_silent", "ratio_missense", "ratio_trunc"])

    if CFG["USE_HOTSPOTS"] and keep_tokens:
        H = np.zeros((n, len(keep_tokens)), np.float32)
        for r, nm in zip(ex["tok_rows"], ex["tok_names"]):
            j = tok_idx.get(nm)
            if j is not None:
                H[r, j] = 1
        add(H, [f"hs_{t}" for t in keep_tokens])
    if CFG["USE_COMBOS"]:
        C, cn = combo_features(ex)
        add(C, [f"cb_{k}" for k in cn])
    if CFG["USE_AA_SUBS"] and len(aa_keep):
        A = ex["aa_cnt"][:, aa_keep] / (tc[:, TYPE_IDX["missense"]] + 1)[:, None]
        add(A, [f"aa_{AA[k // 20]}>{AA[k % 20]}" for k in aa_keep])

    if CFG["USE_PATHWAYS"]:
        P = np.c_[[mut[:, idx].sum(1) if idx else np.zeros(n) for idx in PW_IDX.values()]].T
        add(P, [f"pw_{k}" for k in PW_IDX])

    return np.hstack(blocks), names

X, feat_names = compose(ex_tr)
X_te, _ = compose(ex_te)
print("X", X.shape, "X_te", X_te.shape)

유전자 4171/4384, 핫스팟 토큰 1795, 아미노산쌍 158
X (6201, 6174) X_te (2546, 6174)


## 4. 5-fold CV (XGBoost + Logistic Regression + ExtraTrees)

※ XGB는 검증 fold로 early stopping을 하므로 OOF 점수가 아주 조금 낙관적일 수 있습니다 (모델 간 비교용으로는 충분).


In [11]:
from sklearn.ensemble import ExtraTreesClassifier

def run_cv(X, X_te, verbose=True):
    """반환: oof(dict 모델명->OOF 확률), te(dict 모델명->test 확률, fold 평균), imp(XGB 중요도)"""
    skf = StratifiedKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"])
    names = ["xgb", "lr"] + (["et"] if CFG["USE_ET"] else [])
    oof = {k: np.zeros((len(y), n_class)) for k in names}
    te = {k: np.zeros((len(X_te), n_class)) for k in names}
    imp = np.zeros(X.shape[1])

    for fold, (tr, va) in enumerate(skf.split(X, y)):
        assert len(np.unique(y[tr])) == n_class, "일부 클래스가 train fold에 없습니다 (N_FOLDS를 줄이세요)"
        sw = compute_sample_weight("balanced", y[tr]) if CFG["BALANCE"] else None

        m = xgb.XGBClassifier(
            objective="multi:softprob", n_estimators=CFG["XGB_N_EST"], learning_rate=0.08,
            max_depth=5, subsample=0.8, colsample_bytree=0.3, tree_method="hist",
            eval_metric="mlogloss", early_stopping_rounds=50, random_state=CFG["SEED"], n_jobs=-1,
        )
        m.fit(X[tr], y[tr], sample_weight=sw, eval_set=[(X[va], y[va])], verbose=False)
        oof["xgb"][va] = m.predict_proba(X[va]); te["xgb"] += m.predict_proba(X_te) / CFG["N_FOLDS"]
        imp += m.feature_importances_ / CFG["N_FOLDS"]

        lr = make_pipeline(MaxAbsScaler(), LogisticRegression(
            C=CFG["LR_C"], max_iter=500, class_weight="balanced" if CFG["BALANCE"] else None))
        lr.fit(X[tr], y[tr])
        oof["lr"][va] = lr.predict_proba(X[va]); te["lr"] += lr.predict_proba(X_te) / CFG["N_FOLDS"]

        if CFG["USE_ET"]:
            et = ExtraTreesClassifier(
                n_estimators=CFG["ET_N_EST"], max_features="sqrt", min_samples_leaf=1,
                class_weight="balanced_subsample" if CFG["BALANCE"] else None,
                n_jobs=-1, random_state=CFG["SEED"])
            et.fit(X[tr], y[tr])
            oof["et"][va] = et.predict_proba(X[va]); te["et"] += et.predict_proba(X_te) / CFG["N_FOLDS"]

        if verbose:
            sc = " | ".join(f"{k.upper()}{'(best_iter=%d)' % m.best_iteration if k == 'xgb' else ''} "
                            f"macroF1={f1_score(y[va], oof[k][va].argmax(1), average='macro'):.4f}" for k in names)
            print(f"fold{fold} | {sc}")
    return oof, te, imp

oof, te, imp = run_cv(X, X_te)
oof_xgb, te_xgb, oof_lr, te_lr = oof["xgb"], te["xgb"], oof["lr"], te["lr"]   # 기존 변수명 호환
if CFG["USE_ET"]:
    oof_et, te_et = oof["et"], te["et"]


fold0 | XGB(best_iter=188) macroF1=0.4443 | LR macroF1=0.4217 | ET macroF1=0.3805
fold1 | XGB(best_iter=169) macroF1=0.4591 | LR macroF1=0.4119 | ET macroF1=0.3888
fold2 | XGB(best_iter=173) macroF1=0.4635 | LR macroF1=0.4265 | ET macroF1=0.3917
fold3 | XGB(best_iter=170) macroF1=0.4728 | LR macroF1=0.4186 | ET macroF1=0.3869
fold4 | XGB(best_iter=183) macroF1=0.4612 | LR macroF1=0.4367 | ET macroF1=0.3986


In [12]:
# ---- (선택) 콤보 피처 효과 확인: CFG["RUN_ABLATION"]=True로 켜서 실행 ----
# 같은 방식으로 USE_AA_SUBS / USE_PATHWAYS / USE_HOTSPOTS 도 바꿔가며 비교해 보세요
# (논문 교훈: 데이터가 적을 때 특징을 늘리면 오히려 성능이 떨어질 수 있음).
if CFG["RUN_ABLATION"]:
    keep_flag = CFG["USE_COMBOS"]
    CFG["USE_COMBOS"] = False
    X0, _ = compose(ex_tr); X0_te, _ = compose(ex_te)
    CFG["USE_COMBOS"] = keep_flag
    o0, _, _ = run_cv(X0, X0_te, verbose=False)
    for tag, od in (("콤보 OFF", o0), ("콤보 ON ", oof)):
        for k, p in od.items():
            print(f"{tag} {k.upper():4s}  macroF1={f1_score(y, p.argmax(1), average='macro'):.4f}  acc={accuracy_score(y, p.argmax(1)):.4f}")
else:
    print("ablation 건너뜀 (CFG['RUN_ABLATION']=True로 실행 가능)")


ablation 건너뜀 (CFG['RUN_ABLATION']=True로 실행 가능)


## 5. 평가 · 앙상블 · 오분류 분석

In [13]:
from itertools import product
from scipy.optimize import minimize_scalar
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import log_loss

MODELS = list(oof.keys())
def report(name, p):
    print(f"{name:12s} acc={accuracy_score(y, p.argmax(1)):.4f}  macroF1={f1_score(y, p.argmax(1), average='macro'):.4f}")
for k in MODELS: report(k.upper(), oof[k])

# ---- 확률 보정기 (CUPLR의 isotonic 보정 아이디어) ----
def fit_calibrator(method, P, yy):
    """P(확률), yy(정답)로 보정기를 학습해 '확률 -> 보정 확률' 함수를 돌려준다."""
    if method == "none":
        return lambda Q: Q
    if method == "temp":                       # 온도 스케일링: 모델당 파라미터 1개 -> 희귀 클래스에도 안정적
        L = np.log(np.clip(P, 1e-9, 1))
        def nll(T):
            Z = L / T; Z -= Z.max(1, keepdims=True); Q = np.exp(Z); Q /= Q.sum(1, keepdims=True)
            return -np.log(np.clip(Q[np.arange(len(yy)), yy], 1e-12, 1)).mean()
        T = minimize_scalar(nll, bounds=(0.2, 5.0), method="bounded").x
        def ap(Q):
            Z = np.log(np.clip(Q, 1e-9, 1)) / T; Z -= Z.max(1, keepdims=True)
            R = np.exp(Z); return R / R.sum(1, keepdims=True)
        return ap
    if method == "isotonic":                   # 클래스별 one-vs-rest isotonic + 재정규화
        iso = []
        for c in range(n_class):
            ir = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1)
            ir.fit(P[:, c], (yy == c).astype(float)); iso.append(ir)
        def ap(Q):
            R = np.column_stack([iso[c].predict(Q[:, c]) for c in range(n_class)]) + 1e-6
            return R / R.sum(1, keepdims=True)
        return ap
    raise ValueError(method)

def calibrate_oof(method, P):
    """OOF 확률을 교차 적합으로 보정 (보정기를 학습한 행으로는 평가하지 않음 -> 낙관 편향 방지)."""
    out = np.zeros_like(P)
    for a, b in StratifiedKFold(5, shuffle=True, random_state=CFG["SEED"] + 1).split(P, y):
        out[b] = fit_calibrator(method, P[a], y[a])(P[b])
    return out

# ---- 앙상블 가중치: 모델 수에 맞는 simplex 격자(0.1 간격) ----
def blend(Pd, ws): return sum(w * Pd[k] for w, k in zip(ws, MODELS))
def best_blend(Pd):
    best_ws, best_s = None, -1
    for w in product(np.linspace(0, 1, 11), repeat=len(MODELS) - 1):
        if sum(w) > 1 + 1e-9: continue
        ws = list(w) + [1 - sum(w)]
        s = f1_score(y, blend(Pd, ws).argmax(1), average="macro")
        if s > best_s: best_ws, best_s = ws, s
    return best_ws, best_s

# ---- 보정 방식 비교 -> 이득이 BOOST_MIN_GAIN 이상일 때만 적용 ----
res = {}
for m in CFG["CALIB_METHODS"]:
    Pd = {k: (oof[k] if m == "none" else calibrate_oof(m, oof[k])) for k in MODELS}
    ws, s = best_blend(Pd)
    ens = blend(Pd, ws)
    res[m] = dict(ws=ws, s=s, Pd=Pd, ll=log_loss(y, np.clip(ens / ens.sum(1, keepdims=True), 1e-9, 1), labels=range(n_class)))
    print(f"보정={m:9s} macroF1={s:.4f}  logloss={res[m]['ll']:.4f}  가중치 " +
          ", ".join(f"{k}={w:.1f}" for k, w in zip(MODELS, ws)))

base = res["none"]["s"] if "none" in res else -1
chosen = max(res, key=lambda m: res[m]["s"])
if chosen != "none" and res[chosen]["s"] - base < CFG["BOOST_MIN_GAIN"]:
    chosen = "none"
print(f"\n선택된 보정: {chosen}")

best_ws = res[chosen]["ws"]
oof_ens = blend(res[chosen]["Pd"], best_ws)
# test: 보정기는 전체 OOF로 학습, test 확률(fold 평균)에 적용
te_cal = {k: (te[k] if chosen == "none" else fit_calibrator(chosen, oof[k], y)(te[k])) for k in MODELS}
te_ens = blend(te_cal, best_ws)
best_w = best_ws[0]   # (호환용) XGB 가중치
print("앙상블 가중치:", {k: round(float(w), 1) for k, w in zip(MODELS, best_ws)})
report("ENSEMBLE", oof_ens)

print("\n" + classification_report(y, oof_ens.argmax(1), target_names=le.classes_, digits=3))

cm = confusion_matrix(y, oof_ens.argmax(1)); np.fill_diagonal(cm, 0)
pairs = sorted(((cm[i, j], le.classes_[i], le.classes_[j]) for i in range(n_class) for j in range(n_class) if cm[i, j] > 0), reverse=True)[:10]
print("가장 많이 헷갈리는 쌍 (정답 -> 예측):")
for c, a, b in pairs: print(f"  {a} -> {b}: {c}")

top = np.argsort(imp)[::-1][:20]
print("\nXGB 중요도 상위 20:", [feat_names[i] for i in top])


XGB          acc=0.4451  macroF1=0.4607
LR           acc=0.4133  macroF1=0.4244
ET           acc=0.4011  macroF1=0.3920
보정=none      macroF1=0.4762  logloss=1.7624  가중치 xgb=0.4, lr=0.3, et=0.3
보정=temp      macroF1=0.4776  logloss=1.6934  가중치 xgb=0.6, lr=0.4, et=0.0
보정=isotonic  macroF1=0.4694  logloss=1.5880  가중치 xgb=0.6, lr=0.4, et=0.0

선택된 보정: none
앙상블 가중치: {'xgb': 0.4, 'lr': 0.3, 'et': 0.3}
ENSEMBLE     acc=0.4533  macroF1=0.4762

              precision    recall  f1-score   support

         ACC      0.950     0.792     0.864        72
        BLCA      0.618     0.452     0.522       104
        BRCA      0.594     0.497     0.542       786
        CESC      0.399     0.368     0.383       155
        COAD      0.805     0.722     0.761       223
        DLBC      0.923     0.316     0.471        38
      GBMLGG      0.357     0.278     0.312       461
        HNSC      0.445     0.363     0.400       223
       KIPAN      0.206     0.256     0.228       515
        KIRC      0.1

In [14]:
# ---- COAD 확률 가중 (선택, 자동 판단) ----
# 전체 클래스 prior 보정(EM)은 소수 클래스를 깎아 macroF1을 망가뜨림(0.46 -> 0.21).
# 그래서 COAD 한 클래스만 k배 키우는 방식을 'COAD 22% 분포로 재표집한 OOF'에서 검증하고,
# 이득이 BOOST_MIN_GAIN 미만이면 k=1(미적용)로 둔다.
te_final = te_ens.copy()
if "COAD" in le.classes_:
    coad = int(le.transform(["COAD"])[0])
    def boost(P, k):
        Q = P.copy(); Q[:, coad] *= k
        return Q / Q.sum(1, keepdims=True)

    tr_prior = np.bincount(y) / len(y)
    tgt = tr_prior * ((1 - CFG["COAD_TEST_PRIOR"]) / (1 - tr_prior[coad])); tgt[coad] = CFG["COAD_TEST_PRIOR"]
    rng = np.random.default_rng(CFG["SEED"]); N = 6000
    pools = [np.where(y == c)[0] for c in range(n_class)]
    draws = [np.concatenate([rng.choice(pools[c], max(1, int(round(tgt[c] * N))), replace=True) for c in range(n_class)])
             for _ in range(20)]

    grid = [1.0, 1.25, 1.5, 2.0, 3.0]
    res = {k: float(np.mean([f1_score(y[ix], boost(oof_ens[ix], k).argmax(1), average="macro") for ix in draws])) for k in grid}
    print("COAD 22% 가정 시뮬 macroF1:", {k: round(v, 4) for k, v in res.items()})

    best_k = max(res, key=res.get)
    if res[best_k] - res[1.0] < CFG["BOOST_MIN_GAIN"]:
        best_k = 1.0
    te_final = boost(te_ens, best_k)
    print(f"선택된 k={best_k}  | test COAD 예측 수: {int((te_ens.argmax(1)==coad).sum())} -> {int((te_final.argmax(1)==coad).sum())}"
          f"  (가이드 기준 약 {int(CFG['COAD_TEST_PRIOR']*len(te_ens))}명)")


COAD 22% 가정 시뮬 macroF1: {1.0: 0.4601, 1.25: 0.4598, 1.5: 0.4605, 2.0: 0.4608, 3.0: 0.4601}
선택된 k=1.0  | test COAD 예측 수: 537 -> 537  (가이드 기준 약 560명)


## 5-2. 쌍둥이(형제 라벨) 규칙 (`USE_TWIN`)

- TCGA의 **KIPAN**(= KIRC+KIRP+KICH)과 **GBMLGG**(= GBM+LGG)는 다른 코호트 환자를 다시 묶은 통합 코호트라, 같은 환자가 `KIRC`로 한 번, `KIPAN`으로 한 번 들어 있다 (변이 프로필이 완전히 같음).
- 대회 train/test 분할에서 이 쌍이 갈라지면, test 환자의 쌍둥이는 train에 **짝 라벨**로 들어 있다. 모델은 똑같은 프로필을 train에서 봤으니 train 쪽 라벨(KIRC)을 그대로 찍어서 틀린다.
- 규칙: test 행의 프로필이 train에서 **한 암종으로만** 나온 KIRC/KIPAN/LGG/GBMLGG 프로필과 같으면 **짝 암종**으로 바꾼다 (KIRC↔KIPAN, LGG↔GBMLGG).
- 누수 아님: 찾아보는 사전은 **train만으로** 만들고, test 행은 자기 프로필 하나만 들고 찾아본다 (test끼리 비교하지 않음).
- 검증: 이 노트북의 CV fold는 무작위라 쌍둥이가 학습/검증에 갈라져 들어간다 → test 상황과 비슷하다. 같은 fold 분할로 "학습 부분만 사전으로 써서 검증 부분을 교정"한 결과를 OOF에서 비교한다.


In [15]:
import hashlib

TWIN_SWAP = {"KIPAN": "KIRC", "KIRC": "KIPAN", "GBMLGG": "LGG", "LGG": "GBMLGG"}

def profile_hashes(df):
    """행마다 변이 프로필(유전자 셀 전부)의 지문. 셀이 전부 같으면 지문이 같다."""
    return np.array([hashlib.md5("|".join(r).encode()).hexdigest()[:12]
                     for r in df[gene_cols].to_numpy(dtype=object)])

def twin_override(tr_hash, tr_labels, q_hash):
    """질의 행마다 바꿀 암종 번호(없으면 -1). train 프로필->암종 집합 사전을 만들고 한 행씩 찾아본다."""
    lab = pd.Series(np.asarray(tr_labels)).groupby(np.asarray(tr_hash)).agg(set).to_dict()
    cls = list(le.classes_)
    out = np.full(len(q_hash), -1)
    for i, h in enumerate(q_hash):
        s = lab.get(h, set())
        if len(s) == 1:                      # 쌍이 train에 이미 둘 다 있으면(암종 2개) 건드리지 않음
            only = next(iter(s))
            if only in TWIN_SWAP and TWIN_SWAP[only] in cls:
                out[i] = cls.index(TWIN_SWAP[only])
    return out

def apply_override(P, ov):
    """ov >= 0 인 행은 그 암종 확률을 1로(나머지 0), 나머지 행은 그대로."""
    Q = P.copy(); m = ov >= 0
    Q[m] = 0.0; Q[m, ov[m]] = 1.0
    return Q

# ---- 동작 확인 (작은 예제) ----
_cls_bak = le.classes_
_C = np.array(["BRCA", "GBMLGG", "KIPAN", "KIRC", "LGG"])
le_bak, le = le, type("L", (), {"classes_": _C})()
_ov = twin_override(["a","a","b","c","d","e","e"], ["KIPAN","KIRC","KIRC","LGG","BRCA","KIPAN","KIPAN"],
                    ["a","b","c","d","z","e"])
le = le_bak
assert _ov.tolist() == [-1, 2, 1, -1, -1, 3]   # a: 쌍이 이미 train에 있음 / b: KIRC->KIPAN / c: LGG->GBMLGG / e: KIPAN->KIRC
assert np.allclose(apply_override(np.full((6, 5), 0.2), _ov).sum(1), 1)

# ---- 시뮬레이션: 위 CV와 똑같은 fold 분할, 학습 부분만 사전으로 사용 ----
H_tr, H_te = profile_hashes(train), profile_hashes(test)
lab_tr = train["SUBCLASS"].values
skf = StratifiedKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"])
sim = oof_ens.copy(); n_hit = 0
for tr_i, va_i in skf.split(X, y):
    ov = twin_override(H_tr[tr_i], lab_tr[tr_i], H_tr[va_i])
    sim[va_i] = apply_override(oof_ens[va_i], ov); n_hit += int((ov >= 0).sum())
hit = (sim != oof_ens).any(1)
f0, f1 = (f1_score(y, p.argmax(1), average="macro") for p in (oof_ens, sim))
print(f"쌍둥이 규칙 시뮬: OOF macroF1 {f0:.4f} -> {f1:.4f} (+{f1 - f0:.4f}) | 발동 {n_hit}행 ({n_hit/len(y):.1%}),"
      f" 발동 행 정답률 {(sim[hit].argmax(1) == y[hit]).mean():.3f} (규칙 전 {(oof_ens[hit].argmax(1) == y[hit]).mean():.3f})")

# ---- test 적용: 사전은 train만, test 행은 한 행씩 독립으로 ----
ov_te = twin_override(H_tr, lab_tr, H_te)
print(f"test에서 발동 가능: {(ov_te >= 0).sum()}행 ({(ov_te >= 0).mean():.1%}) ->",
      pd.Series(le.classes_[ov_te[ov_te >= 0]]).value_counts().to_dict())
if CFG["USE_TWIN"] and (f1 - f0) >= CFG["BOOST_MIN_GAIN"]:
    te_final = apply_override(te_final, ov_te)
    print("쌍둥이 규칙 적용함")
else:
    print("쌍둥이 규칙 미적용 (USE_TWIN=False 이거나 시뮬 이득이 BOOST_MIN_GAIN 미만)")


쌍둥이 규칙 시뮬: OOF macroF1 0.4762 -> 0.5476 (+0.0713) | 발동 717행 (11.6%), 발동 행 정답률 0.997 (규칙 전 0.006)
test에서 발동 가능: 222행 (8.7%) -> {'KIRC': 64, 'KIPAN': 58, 'GBMLGG': 52, 'LGG': 48}
쌍둥이 규칙 적용함


## 6. 제출 파일 생성

In [16]:
pred = le.inverse_transform(te_final.argmax(1))
out = sub.copy()
out["SUBCLASS"] = pred
assert len(out) == len(test) and set(out["SUBCLASS"]) <= set(le.classes_)
out.to_csv(f"{D}/submission_v7.csv", index=False, encoding="UTF-8-sig")
np.save(f"{D}/oof_ens_v7.npy", oof_ens); np.save(f"{D}/te_final_v7.npy", te_final)   # 이후 앙상블/후처리용
print(out["SUBCLASS"].value_counts().head(10))


SUBCLASS
COAD      537
STES      401
KIPAN     235
BRCA      175
OV        133
GBMLGG    117
PRAD      104
KIRC       92
SKCM       84
UCEC       76
Name: count, dtype: int64
